In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib.pyplot as plt
import matplotlib

from TrajectoryNet import dataset, eval_utils
from TrajectoryNet.parse import parser
from TrajectoryNet.lib.growth_net import GrowthNet
from TrajectoryNet.lib.viz_scrna import trajectory_to_video, save_vectors

from TrajectoryNet.train_misc import (
    set_cnf_options,
    add_spectral_norm,
    create_regularization_fns,
    build_model_tabular,
)

In [6]:
os.chdir("/ssd/users/Wergillius/Project/PINN_dynamics")

In [ ]:
adata = sc.read_h5ad("data/klein_subset.h5ad")
Fate_bias = pd.read_csv("data/Weinreb/F_obs.csv", index_col=0)
Fate_bias.index = Fate_bias.index.astype(str)

overlapped_cbs = np.intersect1d(Fate_bias.index, adata.obs_names)
overlapped_cb_index = [np.where(adata.obs_names == cb)[0] for cb in overlapped_cbs]


# load config

In [3]:
args = parser.parse_args(
    [
    "--dataset", "data/klein_for_TJN.npz",
    "--embedding_name", "DM",
    "--max_dim", "5",
    "--niters", "3000",
    "--save", "logs/TrajectoryNet/Weinreb_DM_whiten",
    "--growth_path", "logs/TrajectoryNet/Weinreb_DM_whiten/klein_TJN_growth_model.pt",
    "--whiten",
    "--use_growth"
            ]
)

In [7]:
data = dataset.SCData.factory(args.dataset, args)

No velocity found for embedding DM skipping velocity


In [24]:
timepoints = data.get_unique_times()
# Use maximum timepoint to establish integration_times
int_tps = (np.arange(max(timepoints) + 1) + 1.0) * args.time_scale
print(timepoints)
print(int_tps)

[2 4 6]
[0.5 1.  1.5 2.  2.5 3.  3.5]


In [11]:
regularization_fns, regularization_coeffs = create_regularization_fns(args)

# load CNF model

In [15]:
device = 'cuda:0'

In [21]:
model = build_model_tabular(args, data.get_shape()[0], regularization_fns).to(device)
growth_model = torch.load(args.growth_path, map_location=device)

/tmp/ipykernel_2391086/890050661.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  growth_model = torch.load(args.growth_path, map_location=device)


In [23]:
growth_model

GrowthNet(
  (fc1): Linear(in_features=6, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
)

In [20]:
cnf_net = model.chain[0]

In [22]:
cnf_net

CNF(
  (odefunc): RegularizedODEfunc(
    (odefunc): ODEfunc(
      (diffeq): ODEnet(
        (layers): ModuleList(
          (0): ConcatSquashLinear(
            (_layer): Linear(in_features=5, out_features=64, bias=True)
            (_hyper_bias): Linear(in_features=1, out_features=64, bias=False)
            (_hyper_gate): Linear(in_features=1, out_features=64, bias=True)
          )
          (1-2): 2 x ConcatSquashLinear(
            (_layer): Linear(in_features=64, out_features=64, bias=True)
            (_hyper_bias): Linear(in_features=1, out_features=64, bias=False)
            (_hyper_gate): Linear(in_features=1, out_features=64, bias=True)
          )
          (3): ConcatSquashLinear(
            (_layer): Linear(in_features=64, out_features=5, bias=True)
            (_hyper_bias): Linear(in_features=1, out_features=5, bias=False)
            (_hyper_gate): Linear(in_features=1, out_features=5, bias=True)
          )
        )
        (activation_fns): ModuleList(
         

# get tested data

In [38]:
X_test = data.data[overlapped_cb_index]
X_test = X_test[:,0,:]

In [ ]:
adata[overlapped_cbs].obsm[]

In [42]:
X_test

array([[-0.01477837, -0.5707331 , -0.39562136,  0.70434046, -0.02618076],
       [ 0.03048713, -0.49213776, -0.4937449 ,  0.7414075 , -0.02260914],
       [ 0.4927977 , -0.01461635, -1.0832562 , -0.08529068,  0.18291943],
       ...,
       [-0.14377972, -0.49208853, -0.2788845 ,  0.5934794 , -0.03725268],
       [-0.12614654, -0.5663223 , -0.15982564,  0.45698622, -0.03318701],
       [-0.14657943, -0.4768212 , -0.2951031 ,  0.6120738 , -0.03830288]],
      dtype=float32)

In [ ]:
data.sample_index(100)

TypeError: CustomData.sample_index() missing 2 required positional arguments: 'n' and 'label_subset'

In [29]:
logpz = data.base_density()

In [ ]:
logpz()

In [33]:
data.get_data().shape

(47907, 5)

In [25]:
dir(data)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'args',
 'base_density',
 'base_sample',
 'data',
 'data_dict',
 'embedding_name',
 'factory',
 'get_data',
 'get_labels',
 'get_ncells',
 'get_shape',
 'get_times',
 'get_unique_times',
 'get_velocity',
 'has_velocity',
 'known_base_density',
 'labels',
 'leaveout_timepoint',
 'load',
 'ncells',
 'num_timepoints',
 'plot_data',
 'plot_density',
 'plot_paths',
 'plot_velocity',
 'sample_index',
 'use_velocity',
 'val_labels']